In [ ]:
!pip install deap --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 2.7 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from sklearn.datasets import load_iris

from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Load dataset
data = load_iris()
X, y = data.data, data.target
num_features = X.shape[1]

# Define fitness function
def evaluate(individual):
    # Decode individual (binary mask)
    selected_features = [i for i, bit in enumerate(individual) if bit == 1]
    if not selected_features:  # Penalize if no features are selected
        return 0.0,
    X_selected = X[:, selected_features]
    # Evaluate with cross-validation
    model = DecisionTreeClassifier()
    scores = cross_val_score(model, X_selected, y, cv=5)
    return scores.mean(),

# Define GA components
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximize accuracy
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)  # Binary representation
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=num_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)  # Two-point crossover
toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)  # Flip bit mutation
toolbox.register("select", tools.selTournament, tournsize=3)  # Tournament selection

# Parameters
population_size = 20
num_generations = 30
crossover_prob = 0.7
mutation_prob = 0.2

# Run Genetic Algorithm
def main():
    random.seed(42)
    population = toolbox.population(n=population_size)

    # Statistics to track progress
    stats = tools.Statistics(key=lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("max", np.max)

    hof = tools.HallOfFame(1)  # Best individual

    # Run GA
    population, logbook = algorithms.eaSimple(
        population, toolbox,
        cxpb=crossover_prob, mutpb=mutation_prob,
        ngen=num_generations, stats=stats, halloffame=hof, verbose=True
    )

    # Best individual
    best_individual = hof[0]
    selected_features = [i for i, bit in enumerate(best_individual) if bit == 1]
    print("\nBest individual:", best_individual)
    print("Selected features:", selected_features)
    print("Best fitness:", evaluate(best_individual)[0])

if __name__ == "__main__":
    main()


gen	nevals	avg 	max 
0  	20    	0.81	0.96
1  	11    	0.955	0.96
2  	17    	0.954667	0.966667
3  	11    	0.957333	0.966667
4  	12    	0.957667	0.96    
5  	16    	0.957667	0.966667
6  	14    	0.96    	0.966667
7  	18    	0.959   	0.966667
8  	12    	0.959333	0.966667
9  	18    	0.956667	0.966667
10 	17    	0.960333	0.966667
11 	16    	0.959   	0.966667
12 	12    	0.963   	0.966667
13 	19    	0.959333	0.966667
14 	14    	0.960333	0.966667
15 	18    	0.962   	0.966667
16 	14    	0.963   	0.966667
17 	15    	0.96    	0.966667
18 	14    	0.958   	0.966667
19 	17    	0.961333	0.966667
20 	16    	0.962   	0.966667
21 	16    	0.963   	0.966667
22 	17    	0.961333	0.966667
23 	15    	0.959333	0.966667
24 	18    	0.955667	0.966667
25 	17    	0.958667	0.966667
26 	16    	0.954333	0.966667
27 	17    	0.961333	0.966667
28 	18    	0.961333	0.966667
29 	18    	0.962667	0.966667
30 	14    	0.962   	0.966667

Best individual: [0, 1, 1, 1]
Selected features: [1, 2, 3]
Best fitness: 0.9600000000000002
